# Deepfake Audio Detection
This notebook covers the full pipeline for training and evaluating a CNN model to classify speech recordings as Genuine or Deepfake.

In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve

# Ensure memory growth for GPU if available
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

## 1. Data Pipeline and Feature Extraction

In [3]:
SAMPLING_RATE = 16000
MAX_DURATION = 3  # seconds
MAX_SAMPLES = SAMPLING_RATE * MAX_DURATION
BATCH_SIZE = 32

def load_and_preprocess_audio(file_path, label):
    audio_binary = tf.io.read_file(file_path)
    audio, sample_rate = tf.audio.decode_wav(audio_binary)
    audio = tf.squeeze(audio, axis=-1)
    
    audio_len = tf.shape(audio)[0]
    if audio_len < MAX_SAMPLES:
        padding = [[0, MAX_SAMPLES - audio_len]]
        audio = tf.pad(audio, padding, "CONSTANT")
    else:
        audio = audio[:MAX_SAMPLES]
        
    stft = tf.signal.stft(audio, frame_length=512, frame_step=256)
    spectrogram = tf.abs(stft)
    spectrogram = tf.math.log(spectrogram + 1e-6)
    spectrogram = tf.expand_dims(spectrogram, -1)
    
    return spectrogram, label

def create_dataset(directory, batch_size=32):
    fake_dir = os.path.join(directory, 'fake')
    real_dir = os.path.join(directory, 'real')
    
    fake_files = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.endswith('.wav')]
    real_files = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.endswith('.wav')]
    
    file_paths = fake_files + real_files
    labels = [0] * len(fake_files) + [1] * len(real_files)
    
    indices = np.arange(len(file_paths))
    np.random.shuffle(indices)
    file_paths = np.array(file_paths)[indices]
    labels = np.array(labels)[indices]
    
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    ds = ds.map(load_and_preprocess_audio, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

## 2. Define Model Architecture

In [ ]:
def build_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        layers.Conv2D(32, 3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        
        layers.Conv2D(64, 3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        
        layers.Conv2D(128, 3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        
        layers.Conv2D(256, 3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(),
        
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

## 3. Train Model

In [ ]:
train_dir = r"C:\mars ai\datasets\for-norm\training"
val_dir = r"C:\mars ai\datasets\for-norm\validation"

train_ds = create_dataset(train_dir, BATCH_SIZE)
val_ds = create_dataset(val_dir, BATCH_SIZE)

for spec, label in train_ds.take(1):
    input_shape = spec.shape[1:]

model = build_model(input_shape)
model.summary()

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
os.makedirs(r"C:\mars ai\ps2\model", exist_ok=True)
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(filepath=r"C:\mars ai\ps2\model\best_model.keras", monitor='val_accuracy', save_best_only=True)

# history = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stopping, model_checkpoint])

## 4. Evaluation

In [4]:
test_dir = r"C:\mars ai\datasets\for-norm\testing"
test_ds = create_dataset(test_dir, BATCH_SIZE)

# Extract true labels for evaluation
y_true = []
for _, labels in test_ds:
    y_true.extend(labels.numpy())
y_true = np.array(y_true)

model = tf.keras.models.load_model(r"C:\mars ai\ps2\model\best_model.keras")
y_pred_probs = model.predict(test_ds).flatten()

# CALIBRATION FIX: Automatically find the perfect threshold using ROC
fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)
fnr = 1 - tpr
eer_idx = np.nanargmin(np.absolute((fnr - fpr)))
eer = fpr[eer_idx]
optimal_threshold = thresholds[eer_idx]

# Make final decisions using the optimized threshold instead of a hardcoded 0.5
y_pred = (y_pred_probs >= optimal_threshold).astype(int)

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print(f"Overall Accuracy: {acc*100:.2f}%")
print(f"EER: {eer*100:.2f}%")
print(f"Optimal Decision Threshold: {optimal_threshold:.4f}")
print(f"F1 Score: {f1*100:.2f}%")
print("Confusion Matrix:")
print(cm)

145/145 ━━━━━━━━━━━━━━━━━━━━ 34s 221ms/step
Overall Accuracy: 94.30%
EER: 5.70%
Optimal Decision Threshold: 1.0000
F1 Score: 94.18%
Confusion Matrix:
[[2235  135]
 [ 129 2135]]
